# 📊 Data Analysis — 10-Week Course
## Week 8: Classification & Clustering

---
### Learning Objectives
By the end of this week, you will be able to:
- Build and interpret decision tree classifiers
- Apply k-means clustering to discover natural groupings
- Evaluate classifiers using accuracy, precision, recall, and F1 score
- Compare multiple classification models and select the best one

### Outline
1. Classification vs Regression vs Clustering
2. Decision Trees
3. k-Nearest Neighbours
4. Evaluation Metrics
5. k-Means Clustering
6. Hierarchical Clustering
7. Model Comparison
8. Lab Exercises
9. Assignment

---
## 1. Classification vs Regression vs Clustering

| Approach | Output | Supervision | Example |
|---|---|---|---|
| **Regression** | Continuous number | Supervised | Predict GDP |
| **Classification** | Category (class label) | Supervised | High / Low LE |
| **Clustering** | Group membership | Unsupervised | Country profiles |

### Classification Workflow
```
1. Define target class (binary or multi-class)
2. Feature selection and engineering
3. Train/test split (and optional cross-validation)
4. Fit classifier
5. Predict on test set
6. Evaluate: accuracy, confusion matrix, precision/recall/F1
7. Compare models and select best
```

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
from scipy.cluster.hierarchy import dendrogram, linkage

from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    roc_curve, auc, silhouette_score
)
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings("ignore")

plt.rcParams.update({
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
})

import os
if os.path.exists("africa_health_cleaned.csv"):
    df = pd.read_csv("africa_health_cleaned.csv")
else:
    np.random.seed(42)
    N = 54
    countries = [
        "Nigeria","Ethiopia","Egypt","DR Congo","Tanzania","South Africa","Kenya",
        "Uganda","Algeria","Sudan","Morocco","Mozambique","Ghana","Angola","Cameroon",
        "Madagascar","Côte d'Ivoire","Niger","Burkina Faso","Mali","Malawi","Zambia",
        "Senegal","Chad","Somalia","Zimbabwe","Guinea","Rwanda","Benin","Burundi",
        "Tunisia","South Sudan","Togo","Sierra Leone","Libya","Congo","Liberia",
        "Mauritania","Eritrea","Namibia","Gambia","Botswana","Gabon","Lesotho",
        "Guinea-Bissau","Equatorial Guinea","Mauritius","Eswatini","Djibouti",
        "Comoros","Cabo Verde","Sao Tome","Seychelles","São Tomé"
    ]
    regions = (["West Africa"]*16+["East Africa"]*14+
               ["North Africa"]*6+["Central Africa"]*8+["Southern Africa"]*10)
    df = pd.DataFrame({
        "country"           : countries,
        "region"            : regions,
        "life_expectancy"   : np.clip(np.random.normal(63, 8, N), 45, 80),
        "infant_mortality"  : np.clip(np.random.exponential(35, N), 5, 120),
        "maternal_mortality": np.clip(np.random.exponential(350, N), 20, 2000),
        "hiv_prevalence"    : np.clip(np.random.exponential(3.5, N), 0.1, 28),
        "health_expenditure": np.clip(np.random.normal(5.2, 2.1, N), 1, 12),
        "gdp_per_capita"    : np.clip(np.exp(np.random.normal(7.2, 1.2, N)), 300, 15000),
        "vaccination_dtp3"  : np.clip(np.random.normal(78, 18, N), 20, 99),
        "water_access"      : np.clip(np.random.normal(68, 22, N), 15, 99),
    })

# Engineer features
df["log_gdp"]      = np.log(df["gdp_per_capita"])
df["log_infant"]   = np.log(df["infant_mortality"])
df["high_le"]      = (df["life_expectancy"] >= df["life_expectancy"].median()).astype(int)
df["le_category"]  = pd.cut(df["life_expectancy"],
                             bins=[0, 58, 66, 100],
                             labels=["Low", "Medium", "High"])
print("Dataset ready:", df.shape)
print("Class balance (high_le):\n", df["high_le"].value_counts())

---
## 2. Decision Trees

A decision tree splits data at feature thresholds to minimise **impurity** (Gini index or Entropy).

| Hyperparameter | Effect |
|---|---|
| `max_depth` | Limits tree depth → prevents overfitting |
| `min_samples_leaf` | Minimum samples in a leaf → smoother boundaries |
| `criterion` | `gini` or `entropy` — measure of impurity |

In [ ]:
features = ["log_gdp", "vaccination_dtp3", "water_access",
            "health_expenditure", "log_infant"]
X = df[features]
y = df["high_le"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

# Fit decision tree
dt = DecisionTreeClassifier(max_depth=4, random_state=42)
dt.fit(X_train, y_train)

y_pred_dt = dt.predict(X_test)
acc_dt = accuracy_score(y_test, y_pred_dt)
cv_dt  = cross_val_score(dt, X, y, cv=5).mean()

print(f"Decision Tree (depth=4)")
print(f"  Test Accuracy : {acc_dt:.4f}")
print(f"  5-fold CV Acc : {cv_dt:.4f}")
print()
print("Classification Report:")
print(classification_report(y_test, y_pred_dt, target_names=["Low LE", "High LE"]))

# Feature importance
fi = pd.Series(dt.feature_importances_, index=features).sort_values(ascending=False)
print("Feature Importances:")
for feat, imp in fi.items():
    bar = "█" * int(imp * 40)
    print(f"  {feat:<25} {imp:.4f}  {bar}")

In [ ]:
# Visualise the decision tree
fig, ax = plt.subplots(figsize=(18, 7))
plot_tree(
    dt, feature_names=features,
    class_names=["Low LE", "High LE"],
    filled=True, rounded=True, fontsize=9, ax=ax
)
ax.set_title("Decision Tree — Predict High Life Expectancy",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("week8_tree.png", bbox_inches="tight")
plt.show()

# Text rules
print("\nDecision Tree Text Rules (depth ≤ 3):")
print(export_text(dt, feature_names=features, max_depth=3))

---
## 3. k-Nearest Neighbours (kNN)

kNN classifies a point by the **majority class of its k nearest neighbours** in feature space.

- Distance metric: Euclidean by default
- **k too small** → overfitting (high variance)
- **k too large** → underfitting (high bias)
- Always **scale features** before kNN

In [ ]:
# Scale features for kNN
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

# Find optimal k
k_vals = range(1, 15)
cv_scores = []
for k in k_vals:
    knn = KNeighborsClassifier(n_neighbors=k)
    score = cross_val_score(knn, scaler.fit_transform(X), y, cv=5).mean()
    cv_scores.append(score)

best_k = k_vals[np.argmax(cv_scores)]
print(f"Best k = {best_k}  (CV accuracy = {max(cv_scores):.4f})")

# Fit best kNN
knn_best = KNeighborsClassifier(n_neighbors=best_k)
knn_best.fit(X_train_s, y_train)
acc_knn = accuracy_score(y_test, knn_best.predict(X_test_s))
print(f"Test Accuracy (k={best_k}): {acc_knn:.4f}")

# Plot k vs accuracy
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(list(k_vals), cv_scores, marker="o", color="#3498DB", lw=2, ms=7)
ax.axvline(best_k, color="red", ls="--", lw=1.5,
           label=f"Best k={best_k}")
ax.set_xlabel("k (Number of Neighbours)")
ax.set_ylabel("5-fold CV Accuracy")
ax.set_title("kNN: Choosing Optimal k", fontweight="bold")
ax.legend()
plt.tight_layout()
plt.savefig("week8_knn_k.png", bbox_inches="tight")
plt.show()

---
## 4. Evaluation Metrics

### Confusion Matrix
```
                  Predicted
                Pos    Neg
Actual  Pos    TP     FN
        Neg    FP     TN
```

| Metric | Formula | Meaning |
|---|---|---|
| **Accuracy** | (TP+TN)/Total | Overall correct |
| **Precision** | TP/(TP+FP) | When we predict +, how often are we right? |
| **Recall** | TP/(TP+FN) | What fraction of actual + did we catch? |
| **F1** | 2·P·R/(P+R) | Harmonic mean of Precision and Recall |
| **AUC** | Area under ROC | Overall discriminative ability |

**Use precision** when false positives are costly (e.g., spam filter).  
**Use recall** when false negatives are costly (e.g., disease diagnosis).

In [ ]:
# Compare 4 classifiers
classifiers = {
    "Decision Tree" : DecisionTreeClassifier(max_depth=4, random_state=42),
    "kNN (best k)"  : KNeighborsClassifier(n_neighbors=best_k),
    "Random Forest" : RandomForestClassifier(n_estimators=100, random_state=42),
    "SVM (RBF)"     : SVC(probability=True, random_state=42),
}

results = []
X_s = scaler.fit_transform(X)

for name, clf in classifiers.items():
    if name in ["kNN (best k)", "SVM (RBF)"]:
        X_tr_c, X_te_c = X_train_s, X_test_s
        X_cv_c = X_s
    else:
        X_tr_c, X_te_c = X_train, X_test
        X_cv_c = X

    clf.fit(X_tr_c, y_train)
    y_pred = clf.predict(X_te_c)
    y_prob = clf.predict_proba(X_te_c)[:, 1]

    from sklearn.metrics import precision_score, recall_score, f1_score
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    roc_auc = auc(fpr, tpr)
    cv_acc  = cross_val_score(clf, X_cv_c, y, cv=5).mean()

    results.append({
        "Model"     : name,
        "Test Acc"  : accuracy_score(y_test, y_pred),
        "Precision" : precision_score(y_test, y_pred),
        "Recall"    : recall_score(y_test, y_pred),
        "F1"        : f1_score(y_test, y_pred),
        "ROC AUC"   : roc_auc,
        "5-fold CV" : cv_acc,
    })

results_df = pd.DataFrame(results).set_index("Model")
print("=== Model Comparison ===")
print(results_df.round(4).to_string())
print(f"\nBest model by F1: {results_df['F1'].idxmax()}")

In [ ]:
# ROC curves for all classifiers
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# A — ROC curves
ax = axes[0]
colors_models = ["#3498DB", "#2ECC71", "#E74C3C", "#9B59B6"]
for (name, clf), color in zip(classifiers.items(), colors_models):
    if name in ["kNN (best k)", "SVM (RBF)"]:
        X_tr_c, X_te_c = X_train_s, X_test_s
    else:
        X_tr_c, X_te_c = X_train, X_test
    clf.fit(X_tr_c, y_train)
    y_prob = clf.predict_proba(X_te_c)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=color, lw=2, label=f"{name} (AUC={roc_auc:.3f})")

ax.plot([0, 1], [0, 1], "k--", lw=1)
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("A — ROC Curves", fontweight="bold")
ax.legend(fontsize=8)

# B — Metric comparison bar chart
ax = axes[1]
metrics = ["Test Acc", "Precision", "Recall", "F1"]
x_pos = np.arange(len(metrics))
width = 0.18

for i, (model_name, row) in enumerate(results_df.iterrows()):
    values = [row[m] for m in metrics]
    ax.bar(x_pos + i * width, values, width=width * 0.9,
           color=colors_models[i], alpha=0.85, label=model_name)

ax.set_xticks(x_pos + width * 1.5)
ax.set_xticklabels(metrics)
ax.set_ylabel("Score")
ax.set_ylim(0, 1.1)
ax.set_title("B — Metrics Comparison", fontweight="bold")
ax.legend(fontsize=7)

plt.suptitle("Classifier Comparison — Week 8", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("week8_model_compare.png", bbox_inches="tight")
plt.show()

---
## 5. k-Means Clustering

k-means partitions n observations into k clusters, minimising within-cluster variance.

### Algorithm
1. Initialise k centroids randomly
2. Assign each point to nearest centroid
3. Recompute centroids as cluster means
4. Repeat steps 2–3 until convergence

### Choosing k: Elbow Method & Silhouette Score
- **Elbow**: plot inertia (within-cluster SS) vs k; pick the "elbow"
- **Silhouette**: measures how well each point fits its cluster (range: −1 to +1)

In [ ]:
# Features for clustering
cluster_features = ["log_gdp", "life_expectancy", "infant_mortality",
                     "vaccination_dtp3", "water_access"]
X_clust = df[cluster_features]
scaler_c = StandardScaler()
X_clust_s = scaler_c.fit_transform(X_clust)

# Elbow + Silhouette
k_range = range(2, 10)
inertias, silhouettes = [], []

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_clust_s)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_clust_s, labels))

best_k_clust = list(k_range)[np.argmax(silhouettes)]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(list(k_range), inertias, marker="o", color="#3498DB", lw=2)
axes[0].set_xlabel("k")
axes[0].set_ylabel("Inertia")
axes[0].set_title("Elbow Method", fontweight="bold")

axes[1].plot(list(k_range), silhouettes, marker="s", color="#E74C3C", lw=2)
axes[1].axvline(best_k_clust, color="green", ls="--", lw=1.5,
               label=f"Best k={best_k_clust}")
axes[1].set_xlabel("k")
axes[1].set_ylabel("Silhouette Score")
axes[1].set_title("Silhouette Score", fontweight="bold")
axes[1].legend()

plt.suptitle(f"k-Means: Choosing k (best = {best_k_clust})",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig("week8_kmeans_k.png", bbox_inches="tight")
plt.show()
print(f"Best k = {best_k_clust}  (silhouette = {max(silhouettes):.4f})")

In [ ]:
# Fit k-means with best k
km_final = KMeans(n_clusters=best_k_clust, random_state=42, n_init=10)
df["cluster"] = km_final.fit_predict(X_clust_s)

# PCA for 2D visualisation
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_clust_s)
df["pc1"] = X_pca[:, 0]
df["pc2"] = X_pca[:, 1]

cluster_palette = plt.cm.Set1.colors[:best_k_clust]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# A — PCA scatter coloured by cluster
ax = axes[0]
for i in range(best_k_clust):
    mask = df["cluster"] == i
    ax.scatter(df.loc[mask, "pc1"], df.loc[mask, "pc2"],
               color=cluster_palette[i], label=f"Cluster {i+1}",
               alpha=0.8, edgecolors="white", s=80)
    # Annotate a few countries in each cluster
    for _, row in df[mask].sample(min(2, mask.sum()), random_state=1).iterrows():
        ax.annotate(row["country"], (row["pc1"], row["pc2"]),
                    fontsize=7, xytext=(4, 2), textcoords="offset points")

ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% var)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% var)")
ax.set_title("A — k-Means Clusters (PCA projection)", fontweight="bold")
ax.legend()

# B — Cluster profile heatmap
ax = axes[1]
cluster_profile = df.groupby("cluster")[cluster_features].mean()
cluster_profile_norm = (cluster_profile - cluster_profile.mean()) / cluster_profile.std()
sns.heatmap(cluster_profile_norm.T, annot=cluster_profile.T.round(1),
            cmap="RdYlGn", center=0, ax=ax,
            fmt=".0f", cbar_kws={"shrink": 0.8},
            xticklabels=[f"Cluster {i+1}" for i in range(best_k_clust)])
ax.set_title("B — Cluster Profiles (normalised, annotated=actual mean)",
             fontweight="bold")

plt.suptitle("k-Means Clustering — Africa Health Profiles",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("week8_kmeans_results.png", bbox_inches="tight")
plt.show()

# Cluster composition by region
print("\nCluster × Region distribution:")
print(pd.crosstab(df["cluster"], df["region"]))

---
## 6. Hierarchical Clustering

Hierarchical clustering builds a tree (dendrogram) of nested clusters.  
Unlike k-means, you do **not** need to specify k in advance — you cut the tree at the desired level.

In [ ]:
# Hierarchical clustering dendrogram
Z = linkage(X_clust_s, method="ward")

fig, ax = plt.subplots(figsize=(16, 6))
dendrogram(
    Z,
    labels=df["country"].values,
    leaf_rotation=90,
    leaf_font_size=8,
    color_threshold=6,
    ax=ax
)
ax.axhline(6, color="red", ls="--", lw=1.5, label=f"Cut at 6 → {best_k_clust} clusters")
ax.set_ylabel("Ward Linkage Distance")
ax.set_title("Hierarchical Clustering Dendrogram — Africa Health",
             fontweight="bold", fontsize=12)
ax.legend()
plt.tight_layout()
plt.savefig("week8_dendrogram.png", bbox_inches="tight")
plt.show()

---
## 8. Lab Exercises

### Lab 1: Decision Tree Depth Tuning
Train decision trees with `max_depth` from 1 to 10.  
Plot train accuracy vs test accuracy vs depth to visualise overfitting.

In [ ]:
# Lab 1
depths = range(1, 11)
train_accs, test_accs = [], []
for d in depths:
    dt_d = DecisionTreeClassifier(max_depth=d, random_state=42)
    dt_d.fit(X_train, y_train)
    train_accs.append(accuracy_score(y_train, dt_d.predict(X_train)))
    test_accs.append(accuracy_score(y_test,  dt_d.predict(X_test)))

# TODO: Plot train vs test accuracy across depths

### Lab 2: 3-Class Classification
Use `le_category` (Low/Medium/High) as the target.  
Fit a Random Forest and report the multiclass classification report.

In [ ]:
# Lab 2
y_multi = LabelEncoder().fit_transform(df["le_category"])
# TODO: Random Forest for 3-class LE prediction

### Lab 3: Cluster Profiling
Given the k-means clusters already computed, write a function `describe_cluster(df, cluster_id)` that prints the mean values of all numeric features and lists the 5 most representative countries.

In [ ]:
# Lab 3
def describe_cluster(df, cluster_id):
    # TODO
    pass

for i in range(best_k_clust):
    describe_cluster(df, i)

---
## 9. Assignment — Week 8

**Problem 1 (25 pts):** Build a Random Forest classifier to predict `high_le`.  
Plot feature importances as a horizontal bar chart.  
Compare to the Decision Tree — which features are most important in each?

**Problem 2 (25 pts):** Implement a full k-means pipeline:  
- Elbow + silhouette plot to choose k  
- Fit final model  
- Profile each cluster (mean values + top 3 countries)  
- Assign a descriptive name to each cluster

**Problem 3 (25 pts):** Create a 3-class target: countries split into terciles of `life_expectancy` (Low, Medium, High).  
Run a Decision Tree (depth 5) and print the text rules.  
Which single feature best separates Low from High?

**Problem 4 (25 pts):** Write a `compare_classifiers(X, y, classifiers_dict)` function that:  
- Runs each classifier with 5-fold CV  
- Returns a ranked DataFrame of accuracy, F1, and AUC  
- Generates a bar chart of the results

In [ ]:
# Problem 1
pass  # TODO

In [ ]:
# Problem 2
pass  # TODO

In [ ]:
# Problem 3
pass  # TODO

In [ ]:
# Problem 4
def compare_classifiers(X, y, classifiers_dict):
    # TODO
    pass

---
## Summary — Week 8

| Concept | Key Point |
|---|---|
| Decision tree | Splits on feature thresholds; interpretable; prone to overfitting |
| `max_depth` | Limits depth to prevent overfitting |
| kNN | Classifies by majority vote of k nearest neighbours; scale first |
| Random Forest | Ensemble of trees; robust, high accuracy; less interpretable |
| Accuracy | Good for balanced classes; use F1 for imbalanced |
| Precision / Recall | Trade-off based on cost of FP vs FN |
| AUC-ROC | Overall classifier discrimination ability |
| k-Means | Unsupervised; clusters by distance; choose k with elbow/silhouette |
| Hierarchical | Builds dendrogram; no need to choose k upfront |

**Next:** Week 9 — Advanced Topics (Feature Engineering, Dimensionality Reduction, Case Studies)